In [3]:
# ============================================================
# COMPLETE BREAST CANCER COMPRESSION STUDY
# WITH SUSTAINABILITY/CARBON FOOTPRINT ANALYSIS
# ============================================================
#
# Paper: Sustainable Digital Pathology: JPEG2000 Compression
#        Thresholds for Breast Cancer Histopathology Classification
#
# Key Features:
# - Patient-wise 5-fold cross-validation (NO data leakage)
# - Full compression analysis (CR1 to CR1000)
# - Statistical analysis (95% CI, p-values, effect sizes)
# - Carbon footprint & energy consumption tracking
# - Publication-ready visualizations
#
# ============================================================

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

!pip install codecarbon --quiet

import os
import gc
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import DenseNet169, InceptionV3, DenseNet121
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, roc_curve, precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats
from itertools import combinations
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Carbon tracking
from codecarbon import EmissionsTracker, OfflineEmissionsTracker

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("="*70)
print("🔬 BREAST CANCER COMPRESSION STUDY - FULL PIPELINE")
print("🌱 WITH SUSTAINABILITY ANALYSIS")
print("="*70)
print(f"   Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("   Method: Patient-wise 5-fold Cross-Validation")
print("   Model: Inception-DenseNet169 Attention Hybrid")
print("   Tracking: Energy consumption & CO2 emissions")
print("="*70)


# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CONFIG = {
    # Paths
    'data_dir': '/content/drive/MyDrive/BreakHis_Split',
    'output_dir': '/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL',

    # Training
    'image_size': (224, 224),
    'batch_size': 32,
    'epochs': 30,
    'learning_rate': 0.001,
    'n_folds': 5,
    'early_stopping_patience': 5,
    'reduce_lr_patience': 2,

    # Compression levels
    'cr_levels': ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],
    'cr_labels': ['Original', 'CR15', 'CR25', 'CR50', 'CR100', 'CR500', 'CR1000'],

    # For compression-aware training
    'mixed_cr_levels': ['CR1_original', 'CR15', 'CR25', 'CR50', 'CR100'],

    # Sustainability tracking
    'country_iso_code': 'BGD',  # Bangladesh
    'track_emissions': True,
}

# Create output directories
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'models'), exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'figures'), exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'results'), exist_ok=True)
os.makedirs(os.path.join(CONFIG['output_dir'], 'emissions'), exist_ok=True)

print(f"\n📁 Output directory: {CONFIG['output_dir']}")


# ============================================================
# EMISSIONS TRACKER CLASS
# ============================================================

class SustainabilityTracker:
    """
    Wrapper class for tracking energy consumption and carbon emissions.
    Provides consistent interface for all experiments.
    """

    def __init__(self, project_name, output_dir, country_iso_code='BGD'):
        self.project_name = project_name
        self.output_dir = output_dir
        self.country_iso_code = country_iso_code
        self.tracker = None
        self.all_emissions = []

    def start_tracking(self, task_name):
        """Start tracking emissions for a specific task."""
        try:
            self.tracker = OfflineEmissionsTracker(
                project_name=f"{self.project_name}_{task_name}",
                output_dir=self.output_dir,
                country_iso_code=self.country_iso_code,
                log_level="error"
            )
            self.tracker.start()
            self.current_task = task_name
            self.start_time = datetime.now()
            return True
        except Exception as e:
            print(f"⚠️ Emissions tracking error: {e}")
            self.tracker = None
            return False

    def stop_tracking(self):
        """Stop tracking and return emissions data."""
        if self.tracker is None:
            return None

        try:
            emissions = self.tracker.stop()
            end_time = datetime.now()
            duration = (end_time - self.start_time).total_seconds()

            # Get detailed data
            emissions_data = {
                'task': self.current_task,
                'timestamp': end_time.isoformat(),
                'duration_seconds': duration,
                'duration_hours': duration / 3600,
                'emissions_kg_co2': emissions if emissions else 0,
                'emissions_g_co2': (emissions * 1000) if emissions else 0,
            }

            # Try to get energy data from tracker
            if hasattr(self.tracker, '_total_energy'):
                emissions_data['energy_kwh'] = self.tracker._total_energy.kWh
            else:
                # Estimate based on typical Colab GPU power (~150W for T4)
                emissions_data['energy_kwh'] = (duration / 3600) * 0.15  # Estimated

            self.all_emissions.append(emissions_data)
            self.tracker = None

            return emissions_data

        except Exception as e:
            print(f"⚠️ Error stopping tracker: {e}")
            self.tracker = None
            return None

    def get_summary(self):
        """Get summary of all tracked emissions."""
        if not self.all_emissions:
            return None

        df = pd.DataFrame(self.all_emissions)

        summary = {
            'total_duration_hours': df['duration_hours'].sum(),
            'total_emissions_kg_co2': df['emissions_kg_co2'].sum(),
            'total_energy_kwh': df['energy_kwh'].sum(),
            'n_tasks': len(df),
            'avg_emissions_per_task_g': df['emissions_g_co2'].mean(),
            'detailed_emissions': df.to_dict('records'),
        }

        return summary

    def save_emissions(self, filename='emissions_log.csv'):
        """Save emissions data to CSV."""
        if self.all_emissions:
            df = pd.DataFrame(self.all_emissions)
            filepath = os.path.join(self.output_dir, filename)
            df.to_csv(filepath, index=False)
            print(f"💾 Emissions data saved to: {filepath}")
            return filepath
        return None


# Initialize global sustainability tracker
sustainability_tracker = SustainabilityTracker(
    project_name="BreakHis_Compression",
    output_dir=os.path.join(CONFIG['output_dir'], 'emissions'),
    country_iso_code=CONFIG['country_iso_code']
)


# ============================================================
# PRE-DOWNLOAD WEIGHTS
# ============================================================

print("\n" + "="*70)
print("📥 PRE-DOWNLOADING MODEL WEIGHTS")
print("="*70)

for name, fn in [('DenseNet121', DenseNet121), ('DenseNet169', DenseNet169),
                  ('InceptionV3', InceptionV3)]:
    print(f"  {name}...", end=" ", flush=True)
    _ = fn(weights='imagenet', include_top=False)
    print("✅")

print("\n✅ All weights cached!")
tf.keras.backend.clear_session()
gc.collect()


# ============================================================
# DATA AUGMENTATION
# ============================================================

data_augmentation = keras.Sequential([
    layers.RandomRotation(0.15),
    layers.RandomFlip("horizontal"),
    layers.RandomFlip("vertical"),
    layers.RandomZoom(0.15),
], name='augmentation')


# ============================================================
# MODEL ARCHITECTURES
# ============================================================

class ChannelAttention(layers.Layer):
    """Channel attention for feature weighting."""
    def __init__(self, ratio=4, **kwargs):
        super().__init__(**kwargs)
        self.ratio = ratio

    def build(self, input_shape):
        channel = input_shape[-1]
        self.shared_dense1 = layers.Dense(channel // self.ratio, activation='relu')
        self.shared_dense2 = layers.Dense(channel, activation='sigmoid')
        super().build(input_shape)

    def call(self, x):
        avg_pool = tf.reduce_mean(x, axis=1, keepdims=True)
        max_pool = tf.reduce_max(x, axis=1, keepdims=True)
        avg_out = self.shared_dense2(self.shared_dense1(avg_pool))
        max_out = self.shared_dense2(self.shared_dense1(max_pool))
        attention = avg_out + max_out
        return x * attention

    def get_config(self):
        config = super().get_config()
        config.update({'ratio': self.ratio})
        return config


def build_inception_densenet_attention(input_shape=(224, 224, 3)):
    """
    MAIN MODEL: Inception-DenseNet169 Attention Fusion

    Combines:
    - InceptionV3: Multi-scale feature extraction
    - DenseNet169: Fine-grained texture extraction
    - Attention: Learns optimal feature weighting
    """
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # Branch 1: InceptionV3 (multi-scale)
    inception = InceptionV3(weights='imagenet', include_top=False, pooling='avg')
    inception.trainable = False
    x1 = inception(x)

    # Branch 2: DenseNet169 (texture)
    densenet169 = DenseNet169(weights='imagenet', include_top=False, pooling='avg')
    densenet169.trainable = False
    x2 = densenet169(x)

    # Project to same dimension
    x1 = layers.Dense(512, activation='relu', name='inception_proj')(x1)
    x1 = layers.BatchNormalization()(x1)

    x2 = layers.Dense(512, activation='relu', name='densenet_proj')(x2)
    x2 = layers.BatchNormalization()(x2)

    # Stack for attention
    x1_exp = layers.Reshape((1, 512))(x1)
    x2_exp = layers.Reshape((1, 512))(x2)
    stacked = layers.Concatenate(axis=1)([x1_exp, x2_exp])

    # Channel attention
    attended = ChannelAttention(ratio=4)(stacked)

    # Flatten
    fused = layers.Flatten()(attended)

    # Classification head
    x = layers.BatchNormalization()(fused)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='Inception_DenseNet169_Attention')


def build_densenet169(input_shape=(224, 224, 3)):
    """Baseline: DenseNet169 alone."""
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)

    base = DenseNet169(weights='imagenet', include_top=False, pooling='avg')
    base.trainable = False

    x = base(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    return Model(inputs, outputs, name='DenseNet169')


# ============================================================
# DATA LOADING FUNCTIONS
# ============================================================

def get_patient_data(data_dir, cr_level='CR1_original'):
    """Load patient data for a specific compression level."""
    images = []

    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(data_dir, cr_level, split)
        if not os.path.exists(split_dir):
            continue

        for class_idx, class_name in enumerate(['benign', 'malignant']):
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(class_dir):
                continue

            for mag in os.listdir(class_dir):
                mag_dir = os.path.join(class_dir, mag)
                if not os.path.isdir(mag_dir):
                    continue

                for filename in os.listdir(mag_dir):
                    if filename.endswith('.png'):
                        parts = filename.replace('.png', '').split('-')
                        patient_id = f"{parts[1]}-{parts[2]}" if len(parts) >= 3 else 'unknown'

                        images.append({
                            'filepath': os.path.join(mag_dir, filename),
                            'filename': filename,
                            'class_name': class_name,
                            'class_idx': class_idx,
                            'patient_id': patient_id,
                            'magnification': mag,
                            'cr_level': cr_level,
                        })

    return pd.DataFrame(images)


def get_file_size_stats(image_df):
    """Calculate file size statistics for a dataset."""
    file_sizes = []
    for filepath in image_df['filepath'].values[:100]:  # Sample 100 files
        if os.path.exists(filepath):
            file_sizes.append(os.path.getsize(filepath))

    if file_sizes:
        return {
            'mean_size_kb': np.mean(file_sizes) / 1024,
            'total_size_mb': np.sum(file_sizes) / (1024 * 1024) * len(image_df) / 100,
            'n_sampled': len(file_sizes),
        }
    return None


def create_dataset(filepaths, labels, batch_size, shuffle=True, augment=False):
    """Create tf.data.Dataset from file paths."""

    def load_image(filepath, label):
        img = tf.io.read_file(filepath)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, CONFIG['image_size'])
        img = tf.cast(img, tf.float32) / 255.0
        return img, label

    dataset = tf.data.Dataset.from_tensor_slices((filepaths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(filepaths), seed=SEED)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset


# ============================================================
# METRICS COMPUTATION
# ============================================================

def compute_metrics(y_true, y_pred_proba, threshold=0.5):
    """Compute comprehensive metrics."""
    y_pred = (y_pred_proba >= threshold).astype(int)

    if len(np.unique(y_true)) < 2:
        return None

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'f1_score': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'auc_roc': roc_auc_score(y_true, y_pred_proba),
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
    }


# ============================================================
# STATISTICAL ANALYSIS FUNCTIONS
# ============================================================

def compute_95_ci(scores):
    """Compute 95% confidence interval."""
    n = len(scores)
    mean = np.mean(scores)
    std = np.std(scores, ddof=1)
    se = std / np.sqrt(n)
    ci = 1.96 * se
    return mean, ci, (mean - ci, mean + ci)


def paired_ttest(scores1, scores2):
    """Paired t-test between two conditions."""
    if len(scores1) != len(scores2) or len(scores1) < 2:
        return {'p_value': 1.0, 'significant': False}

    t_stat, p_value = stats.ttest_rel(scores1, scores2)
    diff = np.mean(scores1) - np.mean(scores2)

    # Cohen's d
    pooled_std = np.sqrt((np.std(scores1, ddof=1)**2 + np.std(scores2, ddof=1)**2) / 2)
    cohens_d = diff / pooled_std if pooled_std > 0 else 0

    return {
        'mean_diff': diff,
        't_statistic': t_stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'significant_005': p_value < 0.05,
    }


def wilcoxon_test(scores1, scores2):
    """Wilcoxon signed-rank test (non-parametric alternative)."""
    try:
        stat, p_value = stats.wilcoxon(scores1, scores2)
        return {'statistic': stat, 'p_value': p_value}
    except:
        return {'statistic': None, 'p_value': 1.0}


# ============================================================
# 5-FOLD CROSS-VALIDATION TRAINING WITH EMISSIONS TRACKING
# ============================================================

def train_model_cv(build_fn, image_df, model_name, cr_level, save_models=True):
    """
    Train model with 5-fold patient-wise CV.
    Includes energy consumption and CO2 emissions tracking.
    """

    print(f"\n{'='*60}")
    print(f"🔧 Training: {model_name} on {cr_level}")
    print(f"{'='*60}")

    # Get file size stats for this CR level
    file_stats = get_file_size_stats(image_df)
    if file_stats:
        print(f"📁 Avg file size: {file_stats['mean_size_kb']:.1f} KB")

    # Get patient-level labels
    patient_classes = image_df.groupby('patient_id')['class_idx'].first().reset_index()
    patients = patient_classes['patient_id'].values
    labels = patient_classes['class_idx'].values

    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=SEED)

    fold_results = []
    fold_emissions = []
    all_y_true = []
    all_y_pred = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(patients, labels)):
        print(f"\n┌─ FOLD {fold + 1}/{CONFIG['n_folds']} {'─'*45}┐")

        # Start emissions tracking for this fold
        task_name = f"{model_name}_{cr_level}_fold{fold+1}"
        tracking_started = sustainability_tracker.start_tracking(task_name)
        if tracking_started:
            print(f"│ 🌱 Emissions tracking: ACTIVE")

        train_patients = set(patients[train_idx])
        test_patients = set(patients[test_idx])

        # Split train into train/val (85%/15%)
        train_patient_list = list(train_patients)
        np.random.seed(SEED + fold)
        np.random.shuffle(train_patient_list)
        n_train = int(len(train_patient_list) * 0.85)
        val_patients = set(train_patient_list[n_train:])
        train_patients = set(train_patient_list[:n_train])

        # Get dataframes
        train_df = image_df[image_df['patient_id'].isin(train_patients)]
        val_df = image_df[image_df['patient_id'].isin(val_patients)]
        test_df = image_df[image_df['patient_id'].isin(test_patients)]

        print(f"│ Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
        print(f"│ Patients - Train: {len(train_patients)} | Val: {len(val_patients)} | Test: {len(test_patients)}")

        # Class weights
        class_weights = compute_class_weight(
            'balanced',
            classes=np.unique(train_df['class_idx']),
            y=train_df['class_idx'].values
        )
        class_weight_dict = {0: class_weights[0], 1: class_weights[1]}

        # Create datasets
        train_ds = create_dataset(
            train_df['filepath'].values,
            train_df['class_idx'].values,
            CONFIG['batch_size'],
            shuffle=True
        )
        val_ds = create_dataset(
            val_df['filepath'].values,
            val_df['class_idx'].values,
            CONFIG['batch_size'],
            shuffle=False
        )

        # Build model
        print(f"│ Building model...")
        model = build_fn()

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=CONFIG['learning_rate']),
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        # Callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_loss',
                patience=CONFIG['early_stopping_patience'],
                restore_best_weights=True,
                verbose=0
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=CONFIG['reduce_lr_patience'],
                verbose=0
            )
        ]

        print(f"└─ Training {'─'*47}┘")

        # Train
        train_start = datetime.now()
        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=CONFIG['epochs'],
            class_weight=class_weight_dict,
            callbacks=callbacks,
            verbose=1
        )
        train_end = datetime.now()
        train_duration = (train_end - train_start).total_seconds()

        epochs_trained = len(history.history['loss'])
        print(f"\n✅ Trained {epochs_trained} epochs in {train_duration/60:.1f} minutes")

        # Evaluate
        print(f"📊 Evaluating...")
        test_ds = create_dataset(
            test_df['filepath'].values,
            test_df['class_idx'].values,
            CONFIG['batch_size'],
            shuffle=False
        )

        y_pred_proba = model.predict(test_ds, verbose=0).flatten()
        y_true = test_df['class_idx'].values

        metrics = compute_metrics(y_true, y_pred_proba)

        # Stop emissions tracking
        emissions_data = sustainability_tracker.stop_tracking()

        if metrics:
            metrics['fold'] = fold + 1
            metrics['model'] = model_name
            metrics['cr_level'] = cr_level
            metrics['epochs_trained'] = epochs_trained
            metrics['n_train'] = len(train_df)
            metrics['n_test'] = len(test_df)
            metrics['train_duration_sec'] = train_duration

            # Add emissions data
            if emissions_data:
                metrics['energy_kwh'] = emissions_data.get('energy_kwh', 0)
                metrics['co2_emissions_kg'] = emissions_data.get('emissions_kg_co2', 0)
                metrics['co2_emissions_g'] = emissions_data.get('emissions_g_co2', 0)
                fold_emissions.append(emissions_data)

            # Add file size info
            if file_stats:
                metrics['avg_file_size_kb'] = file_stats['mean_size_kb']

            fold_results.append(metrics)

            all_y_true.extend(y_true)
            all_y_pred.extend(y_pred_proba)

            print(f"\n┌─ FOLD {fold + 1} RESULTS {'─'*40}┐")
            print(f"│ Accuracy:  {metrics['accuracy']:.4f}")
            print(f"│ F1-Score:  {metrics['f1_score']:.4f}")
            print(f"│ AUC-ROC:   {metrics['auc_roc']:.4f}")
            print(f"│ MCC:       {metrics['mcc']:.4f}")
            print(f"│ Sens/Spec: {metrics['recall']:.4f} / {metrics['specificity']:.4f}")
            if emissions_data:
                print(f"│ 🌱 Energy:  {emissions_data.get('energy_kwh', 0):.4f} kWh")
                print(f"│ 🌱 CO2:     {emissions_data.get('emissions_g_co2', 0):.2f} g")
            print(f"└{'─'*55}┘")

        # Save model if requested
        if save_models:
            model_path = os.path.join(
                CONFIG['output_dir'], 'models',
                f'{model_name}_{cr_level}_fold{fold+1}.keras'
            )
            model.save(model_path)

        # Cleanup
        del model, train_ds, val_ds, test_ds
        gc.collect()
        tf.keras.backend.clear_session()

    # Summary with 95% CI
    if fold_results:
        accs = [r['accuracy'] for r in fold_results]
        f1s = [r['f1_score'] for r in fold_results]
        aucs = [r['auc_roc'] for r in fold_results]
        mccs = [r['mcc'] for r in fold_results]

        acc_mean, acc_ci, _ = compute_95_ci(accs)
        f1_mean, f1_ci, _ = compute_95_ci(f1s)
        auc_mean, auc_ci, _ = compute_95_ci(aucs)

        # Emissions summary
        total_energy = sum([r.get('energy_kwh', 0) for r in fold_results])
        total_co2 = sum([r.get('co2_emissions_g', 0) for r in fold_results])

        print(f"\n╔═ {model_name} on {cr_level} - SUMMARY {'═'*20}╗")
        print(f"║ Accuracy: {acc_mean:.4f} ± {acc_ci:.4f} (95% CI)")
        print(f"║ F1-Score: {f1_mean:.4f} ± {f1_ci:.4f} (95% CI)")
        print(f"║ AUC-ROC:  {auc_mean:.4f} ± {auc_ci:.4f} (95% CI)")
        print(f"║ MCC:      {np.mean(mccs):.4f} ± {np.std(mccs):.4f}")
        print(f"╠{'═'*55}╣")
        print(f"║ 🌱 SUSTAINABILITY METRICS")
        print(f"║ Total Energy:    {total_energy:.4f} kWh")
        print(f"║ Total CO2:       {total_co2:.2f} g ({total_co2/1000:.4f} kg)")
        print(f"║ CO2 per fold:    {total_co2/CONFIG['n_folds']:.2f} g")
        if acc_mean > 0:
            print(f"║ CO2 per accuracy: {total_co2/(acc_mean*100):.3f} g per %")
        print(f"╚{'═'*55}╝")

    return fold_results, (all_y_true, all_y_pred), fold_emissions


# ============================================================
# ROBUSTNESS EVALUATION (Train on one CR, test on all)
# ============================================================

def evaluate_robustness(model_path, test_df):
    """Evaluate a trained model on a test dataset."""
    model = keras.models.load_model(
        model_path,
        custom_objects={'ChannelAttention': ChannelAttention}
    )

    test_ds = create_dataset(
        test_df['filepath'].values,
        test_df['class_idx'].values,
        CONFIG['batch_size'],
        shuffle=False
    )

    y_pred_proba = model.predict(test_ds, verbose=0).flatten()
    y_true = test_df['class_idx'].values

    metrics = compute_metrics(y_true, y_pred_proba)

    del model
    gc.collect()
    tf.keras.backend.clear_session()

    return metrics


# ============================================================
# MAIN EXPERIMENTS
# ============================================================

def run_experiment_1_adaptation(build_fn, model_name):
    """
    EXPERIMENT 1: ADAPTATION STUDY
    Train and test on the same compression level.
    Shows how well models adapt to each compression ratio.
    Tracks energy consumption for each CR level.
    """
    print("\n" + "="*70)
    print("📊 EXPERIMENT 1: ADAPTATION STUDY")
    print("   Train and test on SAME compression level")
    print("   🌱 Tracking energy consumption per compression level")
    print("="*70)

    all_results = []
    all_emissions = []

    for cr_level in CONFIG['cr_levels']:
        print(f"\n{'#'*60}")
        print(f"# Processing: {cr_level}")
        print(f"{'#'*60}")

        # Load data for this CR level
        image_df = get_patient_data(CONFIG['data_dir'], cr_level)
        print(f"  Loaded {len(image_df):,} images from {image_df['patient_id'].nunique()} patients")

        # Train and evaluate
        fold_results, _, fold_emissions = train_model_cv(
            build_fn, image_df, model_name, cr_level, save_models=True
        )

        all_results.extend(fold_results)
        all_emissions.extend(fold_emissions)

        # Save intermediate results
        results_df = pd.DataFrame(all_results)
        results_df.to_csv(
            os.path.join(CONFIG['output_dir'], 'results', 'experiment1_adaptation.csv'),
            index=False
        )

        # Save emissions data
        if all_emissions:
            emissions_df = pd.DataFrame(all_emissions)
            emissions_df.to_csv(
                os.path.join(CONFIG['output_dir'], 'emissions', 'experiment1_emissions.csv'),
                index=False
            )

    return pd.DataFrame(all_results), pd.DataFrame(all_emissions) if all_emissions else None


def run_experiment_2_robustness(build_fn, model_name):
    """
    EXPERIMENT 2: ROBUSTNESS STUDY
    Train on original images only, test on ALL compression levels.
    Shows how robust models are to unseen compression.
    """
    print("\n" + "="*70)
    print("📊 EXPERIMENT 2: ROBUSTNESS STUDY")
    print("   Train on ORIGINAL, test on ALL compression levels")
    print("="*70)

    # First, train on original images
    cr_original = 'CR1_original'
    image_df_original = get_patient_data(CONFIG['data_dir'], cr_original)

    print(f"\n📥 Training on {cr_original}...")
    fold_results_original, _, fold_emissions = train_model_cv(
        build_fn, image_df_original, model_name, cr_original, save_models=True
    )

    # Now test each fold's model on all compression levels
    all_results = []

    # Get patient splits (same as training)
    patient_classes = image_df_original.groupby('patient_id')['class_idx'].first().reset_index()
    patients = patient_classes['patient_id'].values
    labels = patient_classes['class_idx'].values
    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=SEED)

    for fold, (train_idx, test_idx) in enumerate(skf.split(patients, labels)):
        test_patients = set(patients[test_idx])

        print(f"\n┌─ FOLD {fold + 1} - Testing on all CR levels {'─'*20}┐")

        model_path = os.path.join(
            CONFIG['output_dir'], 'models',
            f'{model_name}_{cr_original}_fold{fold+1}.keras'
        )

        for cr_level in CONFIG['cr_levels']:
            # Load test data for this CR level
            image_df_cr = get_patient_data(CONFIG['data_dir'], cr_level)
            test_df = image_df_cr[image_df_cr['patient_id'].isin(test_patients)]

            if len(test_df) == 0:
                continue

            # Get file size stats
            file_stats = get_file_size_stats(test_df)

            # Evaluate
            metrics = evaluate_robustness(model_path, test_df)

            if metrics:
                metrics['fold'] = fold + 1
                metrics['model'] = model_name
                metrics['train_cr'] = cr_original
                metrics['test_cr'] = cr_level

                if file_stats:
                    metrics['avg_file_size_kb'] = file_stats['mean_size_kb']

                all_results.append(metrics)

                print(f"│ {cr_level:12} → Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1_score']:.4f}")

        print(f"└{'─'*55}┘")

    # Save results
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(
        os.path.join(CONFIG['output_dir'], 'results', 'experiment2_robustness.csv'),
        index=False
    )

    return results_df


def run_experiment_3_compression_aware(build_fn, model_name):
    """
    EXPERIMENT 3: COMPRESSION-AWARE TRAINING
    Train on mixed compression levels, test on ALL levels.
    Shows if mixed training improves robustness.
    """
    print("\n" + "="*70)
    print("📊 EXPERIMENT 3: COMPRESSION-AWARE TRAINING")
    print("   Train on MIXED compression levels")
    print(f"   Training CRs: {CONFIG['mixed_cr_levels']}")
    print("="*70)

    # Combine data from multiple compression levels
    all_dfs = []
    for cr_level in CONFIG['mixed_cr_levels']:
        df = get_patient_data(CONFIG['data_dir'], cr_level)
        all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\n📥 Combined dataset: {len(combined_df):,} images")

    # Train on combined data
    fold_results, _, fold_emissions = train_model_cv(
        build_fn, combined_df, f"{model_name}_MixedCR", 'Mixed', save_models=True
    )

    # Test on all compression levels
    all_results = []

    # Get patient splits
    patient_classes = combined_df.groupby('patient_id')['class_idx'].first().reset_index()
    patients = patient_classes['patient_id'].values
    labels = patient_classes['class_idx'].values
    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=SEED)

    for fold, (train_idx, test_idx) in enumerate(skf.split(patients, labels)):
        test_patients = set(patients[test_idx])

        print(f"\n┌─ FOLD {fold + 1} - Testing Mixed-trained model {'─'*15}┐")

        model_path = os.path.join(
            CONFIG['output_dir'], 'models',
            f'{model_name}_MixedCR_Mixed_fold{fold+1}.keras'
        )

        for cr_level in CONFIG['cr_levels']:
            image_df_cr = get_patient_data(CONFIG['data_dir'], cr_level)
            test_df = image_df_cr[image_df_cr['patient_id'].isin(test_patients)]

            if len(test_df) == 0:
                continue

            metrics = evaluate_robustness(model_path, test_df)

            if metrics:
                metrics['fold'] = fold + 1
                metrics['model'] = model_name
                metrics['train_cr'] = 'Mixed'
                metrics['test_cr'] = cr_level
                all_results.append(metrics)

                print(f"│ {cr_level:12} → Acc: {metrics['accuracy']:.4f}, F1: {metrics['f1_score']:.4f}")

        print(f"└{'─'*55}┘")

    # Save results
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(
        os.path.join(CONFIG['output_dir'], 'results', 'experiment3_compression_aware.csv'),
        index=False
    )

    return results_df


# ============================================================
# STATISTICAL ANALYSIS
# ============================================================

def run_statistical_analysis(exp1_df, exp2_df, exp3_df):
    """Run comprehensive statistical analysis."""

    print("\n" + "="*70)
    print("📈 STATISTICAL ANALYSIS")
    print("="*70)

    results = {}

    # 1. Adaptation Study - Compare CR levels
    print("\n1. ADAPTATION: Comparing compression levels")
    print("-"*60)

    adaptation_stats = []
    for cr_level in CONFIG['cr_levels']:
        cr_data = exp1_df[exp1_df['cr_level'] == cr_level]
        if len(cr_data) > 0:
            accs = cr_data['accuracy'].values
            mean, ci, (low, high) = compute_95_ci(accs)

            # Get energy data if available
            energy = cr_data['energy_kwh'].sum() if 'energy_kwh' in cr_data.columns else 0
            co2 = cr_data['co2_emissions_g'].sum() if 'co2_emissions_g' in cr_data.columns else 0

            adaptation_stats.append({
                'CR Level': cr_level,
                'Mean Accuracy': mean,
                '95% CI Lower': low,
                '95% CI Upper': high,
                'Std': np.std(accs, ddof=1),
                'Total Energy (kWh)': energy,
                'Total CO2 (g)': co2,
            })
            print(f"  {cr_level:12}: {mean:.4f} ± {ci:.4f} | Energy: {energy:.4f} kWh | CO2: {co2:.2f} g")

    results['adaptation'] = pd.DataFrame(adaptation_stats)

    # 2. Robustness - Performance degradation
    print("\n2. ROBUSTNESS: Performance degradation")
    print("-"*60)

    robustness_stats = []
    baseline_acc = exp2_df[exp2_df['test_cr'] == 'CR1_original']['accuracy'].mean()

    for cr_level in CONFIG['cr_levels']:
        cr_data = exp2_df[exp2_df['test_cr'] == cr_level]
        if len(cr_data) > 0:
            accs = cr_data['accuracy'].values
            mean = np.mean(accs)
            degradation = baseline_acc - mean

            robustness_stats.append({
                'Test CR': cr_level,
                'Mean Accuracy': mean,
                'Degradation': degradation,
                'Degradation %': degradation * 100,
            })
            print(f"  {cr_level:12}: {mean:.4f} (degradation: {degradation*100:+.2f}%)")

    results['robustness'] = pd.DataFrame(robustness_stats)

    # 3. Statistical significance tests
    print("\n3. STATISTICAL TESTS")
    print("-"*60)

    # Compare Original vs CR500
    if len(exp2_df) > 0:
        original_accs = exp2_df[exp2_df['test_cr'] == 'CR1_original']['accuracy'].values
        cr500_accs = exp2_df[exp2_df['test_cr'] == 'CR500']['accuracy'].values

        if len(original_accs) > 0 and len(cr500_accs) > 0:
            ttest = paired_ttest(original_accs, cr500_accs)
            print(f"\n  Original vs CR500:")
            print(f"    Mean difference: {ttest['mean_diff']:.4f}")
            print(f"    t-statistic: {ttest['t_statistic']:.4f}")
            print(f"    p-value: {ttest['p_value']:.6f}")
            print(f"    Cohen's d: {ttest['cohens_d']:.4f}")
            print(f"    Significant (p<0.05): {'YES ✓' if ttest['significant_005'] else 'NO'}")

            results['ttest_original_vs_cr500'] = ttest

    # 4. Compression-aware vs Original training
    print("\n4. COMPRESSION-AWARE vs ORIGINAL TRAINING")
    print("-"*60)

    if len(exp3_df) > 0 and len(exp2_df) > 0:
        for cr_level in ['CR500', 'CR1000']:
            orig_accs = exp2_df[exp2_df['test_cr'] == cr_level]['accuracy'].values
            mixed_accs = exp3_df[exp3_df['test_cr'] == cr_level]['accuracy'].values

            if len(orig_accs) > 0 and len(mixed_accs) > 0:
                ttest = paired_ttest(mixed_accs, orig_accs)
                print(f"\n  Testing on {cr_level}:")
                print(f"    Original training: {np.mean(orig_accs):.4f}")
                print(f"    Mixed training:    {np.mean(mixed_accs):.4f}")
                print(f"    Improvement:       {ttest['mean_diff']:+.4f}")
                print(f"    p-value:           {ttest['p_value']:.6f}")

    return results


# ============================================================
# SUSTAINABILITY ANALYSIS
# ============================================================

def run_sustainability_analysis(exp1_df, exp1_emissions):
    """Analyze sustainability metrics across compression levels."""

    print("\n" + "="*70)
    print("🌱 SUSTAINABILITY ANALYSIS")
    print("="*70)

    if exp1_emissions is None or len(exp1_emissions) == 0:
        print("  ⚠️ No emissions data available")
        return None

    # Aggregate by CR level
    sustainability_stats = []

    for cr_level in CONFIG['cr_levels']:
        cr_data = exp1_df[exp1_df['cr_level'] == cr_level]

        if len(cr_data) == 0:
            continue

        # Performance metrics
        acc_mean = cr_data['accuracy'].mean()
        f1_mean = cr_data['f1_score'].mean()

        # Energy/emissions metrics
        total_energy = cr_data['energy_kwh'].sum() if 'energy_kwh' in cr_data.columns else 0
        total_co2_g = cr_data['co2_emissions_g'].sum() if 'co2_emissions_g' in cr_data.columns else 0
        total_duration = cr_data['train_duration_sec'].sum() if 'train_duration_sec' in cr_data.columns else 0

        # File size
        avg_file_size = cr_data['avg_file_size_kb'].mean() if 'avg_file_size_kb' in cr_data.columns else 0

        # Efficiency metrics
        co2_per_accuracy = total_co2_g / (acc_mean * 100) if acc_mean > 0 else 0
        energy_per_accuracy = total_energy / (acc_mean * 100) if acc_mean > 0 else 0

        sustainability_stats.append({
            'CR Level': cr_level,
            'Accuracy': acc_mean,
            'F1-Score': f1_mean,
            'Total Energy (kWh)': total_energy,
            'Total CO2 (g)': total_co2_g,
            'Total CO2 (kg)': total_co2_g / 1000,
            'Training Time (hours)': total_duration / 3600,
            'Avg File Size (KB)': avg_file_size,
            'CO2 per Accuracy (g/%)': co2_per_accuracy,
            'Energy per Accuracy (kWh/%)': energy_per_accuracy,
        })

    sustainability_df = pd.DataFrame(sustainability_stats)

    # Print summary
    print("\n📊 SUSTAINABILITY METRICS BY COMPRESSION LEVEL")
    print("-"*80)
    print(f"{'CR Level':<15} {'Accuracy':>10} {'Energy (kWh)':>15} {'CO2 (g)':>12} {'CO2/Acc':>12}")
    print("-"*80)

    for _, row in sustainability_df.iterrows():
        print(f"{row['CR Level']:<15} {row['Accuracy']:>10.4f} {row['Total Energy (kWh)']:>15.4f} {row['Total CO2 (g)']:>12.2f} {row['CO2 per Accuracy (g/%)']:>12.3f}")

    # Calculate savings
    if len(sustainability_df) >= 2:
        original = sustainability_df[sustainability_df['CR Level'] == 'CR1_original'].iloc[0]

        print("\n📈 POTENTIAL SAVINGS vs ORIGINAL")
        print("-"*60)

        for _, row in sustainability_df.iterrows():
            if row['CR Level'] != 'CR1_original':
                acc_diff = (row['Accuracy'] - original['Accuracy']) * 100
                energy_saved = original['Total Energy (kWh)'] - row['Total Energy (kWh)']
                co2_saved = original['Total CO2 (g)'] - row['Total CO2 (g)']

                print(f"  {row['CR Level']}:")
                print(f"    Accuracy change: {acc_diff:+.2f}%")
                print(f"    Energy saved:    {energy_saved:.4f} kWh ({energy_saved/original['Total Energy (kWh)']*100:.1f}%)")
                print(f"    CO2 saved:       {co2_saved:.2f} g ({co2_saved/original['Total CO2 (g)']*100:.1f}%)")

    # Save sustainability data
    sustainability_df.to_csv(
        os.path.join(CONFIG['output_dir'], 'results', 'sustainability_analysis.csv'),
        index=False
    )

    return sustainability_df


# ============================================================
# VISUALIZATION FUNCTIONS
# ============================================================

def create_visualizations(exp1_df, exp2_df, exp3_df, sustainability_df=None):
    """Create publication-ready visualizations including sustainability plots."""

    print("\n" + "="*70)
    print("📊 CREATING VISUALIZATIONS")
    print("="*70)

    fig_dir = os.path.join(CONFIG['output_dir'], 'figures')

    # Set style
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 14
    plt.rcParams['axes.titlesize'] = 14
    plt.rcParams['figure.figsize'] = (10, 6)

    # 1. Adaptation Study - Bar chart with error bars
    print("  Creating Figure 1: Adaptation study...")

    fig, ax = plt.subplots(figsize=(12, 6))

    cr_means = []
    cr_cis = []
    cr_labels = []

    for cr_level in CONFIG['cr_levels']:
        cr_data = exp1_df[exp1_df['cr_level'] == cr_level]
        if len(cr_data) > 0:
            accs = cr_data['accuracy'].values
            mean, ci, _ = compute_95_ci(accs)
            cr_means.append(mean * 100)
            cr_cis.append(ci * 100)
            cr_labels.append(cr_level.replace('CR1_original', 'Original'))

    x = np.arange(len(cr_labels))
    bars = ax.bar(x, cr_means, yerr=cr_cis, capsize=5, color='steelblue', edgecolor='black')

    ax.set_xlabel('Compression Ratio')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Experiment 1: Adaptation Study\n(Train and test on same compression level)')
    ax.set_xticks(x)
    ax.set_xticklabels(cr_labels, rotation=45, ha='right')
    ax.set_ylim([0, 100])

    # Add value labels
    for bar, mean, ci in zip(bars, cr_means, cr_cis):
        ax.annotate(f'{mean:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 5), textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'fig1_adaptation.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(fig_dir, 'fig1_adaptation.pdf'), bbox_inches='tight')
    plt.close()

    # 2. Robustness Study - Line plot
    print("  Creating Figure 2: Robustness study...")

    fig, ax = plt.subplots(figsize=(12, 6))

    rob_means = []
    rob_cis = []

    for cr_level in CONFIG['cr_levels']:
        cr_data = exp2_df[exp2_df['test_cr'] == cr_level]
        if len(cr_data) > 0:
            accs = cr_data['accuracy'].values
            mean, ci, _ = compute_95_ci(accs)
            rob_means.append(mean * 100)
            rob_cis.append(ci * 100)

    x = np.arange(len(CONFIG['cr_labels']))
    ax.errorbar(x, rob_means, yerr=rob_cis, marker='o', markersize=10,
                linewidth=2, capsize=5, color='darkred', label='Original Training')

    # Add mixed training results
    if len(exp3_df) > 0:
        mixed_means = []
        mixed_cis = []
        for cr_level in CONFIG['cr_levels']:
            cr_data = exp3_df[exp3_df['test_cr'] == cr_level]
            if len(cr_data) > 0:
                accs = cr_data['accuracy'].values
                mean, ci, _ = compute_95_ci(accs)
                mixed_means.append(mean * 100)
                mixed_cis.append(ci * 100)

        if len(mixed_means) == len(x):
            ax.errorbar(x, mixed_means, yerr=mixed_cis, marker='s', markersize=10,
                        linewidth=2, capsize=5, color='darkgreen', label='Compression-Aware Training')

    ax.set_xlabel('Test Compression Ratio')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Experiment 2 & 3: Robustness Study\n(Train on original/mixed, test on all compression levels)')
    ax.set_xticks(x)
    ax.set_xticklabels(CONFIG['cr_labels'], rotation=45, ha='right')
    ax.set_ylim([40, 100])
    ax.legend(loc='lower left')
    ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='80% threshold')

    # Highlight critical region
    ax.axvspan(4.5, 6.5, alpha=0.2, color='red', label='Critical degradation')

    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'fig2_robustness.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(fig_dir, 'fig2_robustness.pdf'), bbox_inches='tight')
    plt.close()

    # 3. Heatmap - Metrics across compression levels
    print("  Creating Figure 3: Metrics heatmap...")

    metrics_for_heatmap = ['accuracy', 'precision', 'recall', 'specificity', 'f1_score', 'auc_roc']
    heatmap_data = []

    for cr_level in CONFIG['cr_levels']:
        cr_data = exp2_df[exp2_df['test_cr'] == cr_level]
        if len(cr_data) > 0:
            row = [cr_data[m].mean() for m in metrics_for_heatmap]
            heatmap_data.append(row)

    if len(heatmap_data) > 0:
        fig, ax = plt.subplots(figsize=(10, 8))

        heatmap_df = pd.DataFrame(
            heatmap_data,
            columns=['Accuracy', 'Precision', 'Recall', 'Specificity', 'F1-Score', 'AUC-ROC'],
            index=CONFIG['cr_labels']
        )

        sns.heatmap(heatmap_df, annot=True, fmt='.3f', cmap='RdYlGn',
                    vmin=0.4, vmax=1.0, ax=ax, cbar_kws={'label': 'Score'})

        ax.set_xlabel('Metrics')
        ax.set_ylabel('Test Compression Ratio')
        ax.set_title('Performance Metrics Across Compression Levels\n(Robustness Experiment)')

        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'fig3_metrics_heatmap.png'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(fig_dir, 'fig3_metrics_heatmap.pdf'), bbox_inches='tight')
        plt.close()

    # 4. Box plots - Fold variation
    print("  Creating Figure 4: Fold variation...")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Adaptation
    adaptation_data = []
    for cr_level in CONFIG['cr_levels']:
        cr_data = exp1_df[exp1_df['cr_level'] == cr_level]
        for acc in cr_data['accuracy'].values:
            adaptation_data.append({
                'CR Level': cr_level.replace('CR1_original', 'Original'),
                'Accuracy': acc * 100
            })

    if len(adaptation_data) > 0:
        adapt_df = pd.DataFrame(adaptation_data)
        sns.boxplot(data=adapt_df, x='CR Level', y='Accuracy', ax=axes[0], palette='Blues')
        axes[0].set_title('Adaptation: Accuracy Distribution per CR Level')
        axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
        axes[0].set_ylim([50, 100])

    # Robustness
    robustness_data = []
    for cr_level in CONFIG['cr_levels']:
        cr_data = exp2_df[exp2_df['test_cr'] == cr_level]
        for acc in cr_data['accuracy'].values:
            robustness_data.append({
                'CR Level': cr_level.replace('CR1_original', 'Original'),
                'Accuracy': acc * 100
            })

    if len(robustness_data) > 0:
        rob_df = pd.DataFrame(robustness_data)
        sns.boxplot(data=rob_df, x='CR Level', y='Accuracy', ax=axes[1], palette='Reds')
        axes[1].set_title('Robustness: Accuracy Distribution per CR Level')
        axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
        axes[1].set_ylim([30, 100])

    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'fig4_fold_variation.png'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(fig_dir, 'fig4_fold_variation.pdf'), bbox_inches='tight')
    plt.close()

    # 5. SUSTAINABILITY FIGURE - Accuracy vs CO2
    if sustainability_df is not None and len(sustainability_df) > 0:
        print("  Creating Figure 5: Sustainability analysis...")

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # 5a. Accuracy vs CO2 emissions
        ax1 = axes[0]

        cr_labels_clean = [cr.replace('CR1_original', 'Original') for cr in sustainability_df['CR Level']]
        accuracies = sustainability_df['Accuracy'].values * 100
        co2_values = sustainability_df['Total CO2 (g)'].values

        # Create scatter with size proportional to energy
        scatter = ax1.scatter(
            accuracies, co2_values,
            s=sustainability_df['Total Energy (kWh)'].values * 500,  # Size based on energy
            c=range(len(sustainability_df)),  # Color gradient
            cmap='RdYlGn_r',
            alpha=0.7,
            edgecolors='black'
        )

        # Add labels
        for i, label in enumerate(cr_labels_clean):
            ax1.annotate(label, (accuracies[i], co2_values[i]),
                        xytext=(5, 5), textcoords='offset points', fontsize=10)

        ax1.set_xlabel('Accuracy (%)')
        ax1.set_ylabel('CO2 Emissions (g)')
        ax1.set_title('Accuracy vs Carbon Footprint\n(Bubble size = Energy consumption)')
        ax1.grid(True, alpha=0.3)

        # Add optimal region
        ax1.axhline(y=co2_values[0] * 0.9, color='green', linestyle='--', alpha=0.5)
        ax1.axvline(x=80, color='blue', linestyle='--', alpha=0.5)

        # 5b. Bar chart of CO2 per accuracy point
        ax2 = axes[1]

        co2_per_acc = sustainability_df['CO2 per Accuracy (g/%)'].values
        colors = ['green' if v < co2_per_acc[0] else 'orange' if v < co2_per_acc[0] * 1.5 else 'red'
                  for v in co2_per_acc]

        bars = ax2.bar(range(len(cr_labels_clean)), co2_per_acc, color=colors, edgecolor='black')
        ax2.set_xticks(range(len(cr_labels_clean)))
        ax2.set_xticklabels(cr_labels_clean, rotation=45, ha='right')
        ax2.set_xlabel('Compression Ratio')
        ax2.set_ylabel('CO2 per Accuracy Point (g/%)')
        ax2.set_title('Carbon Efficiency\n(Lower is better)')

        # Add baseline reference
        ax2.axhline(y=co2_per_acc[0], color='gray', linestyle='--', alpha=0.5, label='Original baseline')
        ax2.legend()

        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'fig5_sustainability.png'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(fig_dir, 'fig5_sustainability.pdf'), bbox_inches='tight')
        plt.close()

        # 6. Combined sustainability overview
        print("  Creating Figure 6: Sustainability overview...")

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        # 6a. Accuracy trend
        axes[0, 0].bar(range(len(cr_labels_clean)), accuracies, color='steelblue', edgecolor='black')
        axes[0, 0].set_xticks(range(len(cr_labels_clean)))
        axes[0, 0].set_xticklabels(cr_labels_clean, rotation=45, ha='right')
        axes[0, 0].set_ylabel('Accuracy (%)')
        axes[0, 0].set_title('Classification Accuracy')
        axes[0, 0].axhline(y=80, color='red', linestyle='--', alpha=0.5)

        # 6b. Energy consumption
        energy_values = sustainability_df['Total Energy (kWh)'].values
        axes[0, 1].bar(range(len(cr_labels_clean)), energy_values, color='orange', edgecolor='black')
        axes[0, 1].set_xticks(range(len(cr_labels_clean)))
        axes[0, 1].set_xticklabels(cr_labels_clean, rotation=45, ha='right')
        axes[0, 1].set_ylabel('Energy (kWh)')
        axes[0, 1].set_title('Energy Consumption')

        # 6c. CO2 emissions
        axes[1, 0].bar(range(len(cr_labels_clean)), co2_values, color='forestgreen', edgecolor='black')
        axes[1, 0].set_xticks(range(len(cr_labels_clean)))
        axes[1, 0].set_xticklabels(cr_labels_clean, rotation=45, ha='right')
        axes[1, 0].set_ylabel('CO2 (g)')
        axes[1, 0].set_title('Carbon Emissions')

        # 6d. File size
        file_sizes = sustainability_df['Avg File Size (KB)'].values
        axes[1, 1].bar(range(len(cr_labels_clean)), file_sizes, color='purple', edgecolor='black')
        axes[1, 1].set_xticks(range(len(cr_labels_clean)))
        axes[1, 1].set_xticklabels(cr_labels_clean, rotation=45, ha='right')
        axes[1, 1].set_ylabel('File Size (KB)')
        axes[1, 1].set_title('Average Image File Size')

        plt.suptitle('Sustainability Overview: Compression Impact on Resources', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(os.path.join(fig_dir, 'fig6_sustainability_overview.png'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(fig_dir, 'fig6_sustainability_overview.pdf'), bbox_inches='tight')
        plt.close()

    print("  ✅ All figures saved!")


# ============================================================
# GENERATE RESULTS TABLES FOR PAPER
# ============================================================

def generate_paper_tables(exp1_df, exp2_df, exp3_df, stats_results, sustainability_df=None):
    """Generate LaTeX tables for the paper."""

    print("\n" + "="*70)
    print("📝 GENERATING PAPER TABLES")
    print("="*70)

    tables_dir = os.path.join(CONFIG['output_dir'], 'results')

    # Table 1: Adaptation Results
    print("  Creating Table 1: Adaptation results...")

    table1_data = []
    for cr_level in CONFIG['cr_levels']:
        cr_data = exp1_df[exp1_df['cr_level'] == cr_level]
        if len(cr_data) > 0:
            acc_mean, acc_ci, _ = compute_95_ci(cr_data['accuracy'].values)
            f1_mean, f1_ci, _ = compute_95_ci(cr_data['f1_score'].values)
            auc_mean, auc_ci, _ = compute_95_ci(cr_data['auc_roc'].values)

            table1_data.append({
                'CR Level': cr_level.replace('CR1_original', 'Original'),
                'Accuracy': f"{acc_mean*100:.2f} ± {acc_ci*100:.2f}",
                'F1-Score': f"{f1_mean:.4f} ± {f1_ci:.4f}",
                'AUC-ROC': f"{auc_mean:.4f} ± {auc_ci:.4f}",
            })

    table1_df = pd.DataFrame(table1_data)
    table1_df.to_csv(os.path.join(tables_dir, 'table1_adaptation.csv'), index=False)

    # LaTeX
    latex_table1 = table1_df.to_latex(index=False, escape=False)
    with open(os.path.join(tables_dir, 'table1_adaptation.tex'), 'w') as f:
        f.write(latex_table1)

    # Table 2: Robustness Results
    print("  Creating Table 2: Robustness results...")

    table2_data = []
    baseline = exp2_df[exp2_df['test_cr'] == 'CR1_original']['accuracy'].mean()

    for cr_level in CONFIG['cr_levels']:
        cr_data = exp2_df[exp2_df['test_cr'] == cr_level]
        if len(cr_data) > 0:
            acc_mean = cr_data['accuracy'].mean()
            prec_mean = cr_data['precision'].mean()
            rec_mean = cr_data['recall'].mean()
            spec_mean = cr_data['specificity'].mean()
            degradation = (baseline - acc_mean) * 100

            table2_data.append({
                'Test CR': cr_level.replace('CR1_original', 'Original'),
                'Accuracy': f"{acc_mean*100:.2f}%",
                'Precision': f"{prec_mean:.4f}",
                'Recall': f"{rec_mean:.4f}",
                'Specificity': f"{spec_mean:.4f}",
                'Degradation': f"{degradation:+.2f}%",
            })

    table2_df = pd.DataFrame(table2_data)
    table2_df.to_csv(os.path.join(tables_dir, 'table2_robustness.csv'), index=False)

    # Table 3: Sustainability Results (NEW!)
    if sustainability_df is not None:
        print("  Creating Table 3: Sustainability results...")

        table3_data = []
        for _, row in sustainability_df.iterrows():
            table3_data.append({
                'CR Level': row['CR Level'].replace('CR1_original', 'Original'),
                'Accuracy (%)': f"{row['Accuracy']*100:.2f}",
                'Energy (kWh)': f"{row['Total Energy (kWh)']:.4f}",
                'CO2 (g)': f"{row['Total CO2 (g)']:.2f}",
                'Training Time (h)': f"{row['Training Time (hours)']:.2f}",
                'File Size (KB)': f"{row['Avg File Size (KB)']:.1f}",
                'CO2/Accuracy (g/%)': f"{row['CO2 per Accuracy (g/%)']:.3f}",
            })

        table3_df = pd.DataFrame(table3_data)
        table3_df.to_csv(os.path.join(tables_dir, 'table3_sustainability.csv'), index=False)

        latex_table3 = table3_df.to_latex(index=False, escape=False)
        with open(os.path.join(tables_dir, 'table3_sustainability.tex'), 'w') as f:
            f.write(latex_table3)

    print("  ✅ All tables saved!")


# ============================================================
# MAIN EXECUTION
# ============================================================

def main():
    """Main execution function."""

    start_time = datetime.now()

    print("\n" + "="*70)
    print("🚀 STARTING FULL EXPERIMENT PIPELINE")
    print("🌱 WITH SUSTAINABILITY TRACKING")
    print("="*70)
    print(f"   Start time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

    # Select model
    build_fn = build_inception_densenet_attention
    model_name = "Inception_DenseNet169_Attention"

    print(f"\n📋 Model: {model_name}")
    print(f"📋 Compression levels: {len(CONFIG['cr_levels'])}")
    print(f"📋 Cross-validation folds: {CONFIG['n_folds']}")
    print(f"📋 Emissions tracking: {'ENABLED' if CONFIG['track_emissions'] else 'DISABLED'}")

    # ============ EXPERIMENT 1: ADAPTATION ============
    print("\n" + "#"*70)
    print("# EXPERIMENT 1: ADAPTATION STUDY")
    print("#"*70)

    exp1_results, exp1_emissions = run_experiment_1_adaptation(build_fn, model_name)

    exp1_time = datetime.now()
    print(f"\n⏱️ Experiment 1 completed in: {exp1_time - start_time}")

    # ============ EXPERIMENT 2: ROBUSTNESS ============
    print("\n" + "#"*70)
    print("# EXPERIMENT 2: ROBUSTNESS STUDY")
    print("#"*70)

    exp2_results = run_experiment_2_robustness(build_fn, model_name)

    exp2_time = datetime.now()
    print(f"\n⏱️ Experiment 2 completed in: {exp2_time - exp1_time}")

    # ============ EXPERIMENT 3: COMPRESSION-AWARE ============
    print("\n" + "#"*70)
    print("# EXPERIMENT 3: COMPRESSION-AWARE TRAINING")
    print("#"*70)

    exp3_results = run_experiment_3_compression_aware(build_fn, model_name)

    exp3_time = datetime.now()
    print(f"\n⏱️ Experiment 3 completed in: {exp3_time - exp2_time}")

    # ============ STATISTICAL ANALYSIS ============
    stats_results = run_statistical_analysis(exp1_results, exp2_results, exp3_results)

    # ============ SUSTAINABILITY ANALYSIS ============
    sustainability_df = run_sustainability_analysis(exp1_results, exp1_emissions)

    # ============ VISUALIZATIONS ============
    create_visualizations(exp1_results, exp2_results, exp3_results, sustainability_df)

    # ============ PAPER TABLES ============
    generate_paper_tables(exp1_results, exp2_results, exp3_results, stats_results, sustainability_df)

    # ============ SAVE GLOBAL EMISSIONS ============
    sustainability_tracker.save_emissions('all_emissions.csv')
    emissions_summary = sustainability_tracker.get_summary()

    # ============ FINAL SUMMARY ============
    end_time = datetime.now()

    print("\n" + "="*70)
    print("🎉 EXPERIMENT COMPLETE!")
    print("="*70)
    print(f"   Total runtime: {end_time - start_time}")
    print(f"   Results saved to: {CONFIG['output_dir']}")

    # Print sustainability summary
    if emissions_summary:
        print("\n🌱 SUSTAINABILITY SUMMARY")
        print("-"*50)
        print(f"   Total training time:    {emissions_summary['total_duration_hours']:.2f} hours")
        print(f"   Total energy consumed:  {emissions_summary['total_energy_kwh']:.4f} kWh")
        print(f"   Total CO2 emissions:    {emissions_summary['total_emissions_kg_co2']:.4f} kg")
        print(f"   Avg CO2 per training:   {emissions_summary['avg_emissions_per_task_g']:.2f} g")

    print("\n📁 Output files:")
    print("   - results/experiment1_adaptation.csv")
    print("   - results/experiment2_robustness.csv")
    print("   - results/experiment3_compression_aware.csv")
    print("   - results/sustainability_analysis.csv")
    print("   - results/table1_adaptation.csv")
    print("   - results/table2_robustness.csv")
    print("   - results/table3_sustainability.csv")
    print("   - emissions/all_emissions.csv")
    print("   - figures/fig1_adaptation.png")
    print("   - figures/fig2_robustness.png")
    print("   - figures/fig3_metrics_heatmap.png")
    print("   - figures/fig4_fold_variation.png")
    print("   - figures/fig5_sustainability.png")
    print("   - figures/fig6_sustainability_overview.png")
    print("   - models/*.keras")

    # Save final summary
    summary = {
        'model': model_name,
        'n_folds': CONFIG['n_folds'],
        'cr_levels': CONFIG['cr_levels'],
        'total_runtime': str(end_time - start_time),
        'exp1_time': str(exp1_time - start_time),
        'exp2_time': str(exp2_time - exp1_time),
        'exp3_time': str(exp3_time - exp2_time),
        'sustainability': emissions_summary,
    }

    with open(os.path.join(CONFIG['output_dir'], 'experiment_summary.json'), 'w') as f:
        json.dump(summary, f, indent=2, default=str)

    print("\n✅ All done!")

    return exp1_results, exp2_results, exp3_results, stats_results, sustainability_df


# ============================================================
# RUN
# ============================================================

if __name__ == '__main__':
    exp1_df, exp2_df, exp3_df, stats, sustainability = main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.3/365.3 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.1 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔬 BREAST CANCER COMPRESSION STUDY - FULL PIPELINE
🌱 WITH SUSTAINABILITY ANALYSIS
   Started: 2026-02-27 08:40:38
   Method: Patient-wise 5-fold Cross-Validation
   Model: Inception-DenseNet169 Attention Hybrid
   Tracking: Energy consumption & CO2 emissions

📁 Output directory: /content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL

📥 PRE-DOWNLOADING MODEL WEIGHTS
  DenseNet121... Downloading data from https://storage.googleapis.com/tensorflow/keras-applications/densenet/densenet121_weights_tf_dim_ordering_tf_kernels_notop.h5
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
✅
  DenseNet169... Downloading data from https://storage

[codecarbon INFO @ 08:41:21] offline tracker init
[codecarbon WARNING @ 08:41:21] Multiple instances of codecarbon are allowed to run at the same time.


  Loaded 7,910 images from 81 patients

🔧 Training: Inception_DenseNet169_Attention on CR1_original
📁 Avg file size: 480.9 KB

┌─ FOLD 1/5 ─────────────────────────────────────────────┐
│ 🌱 Emissions tracking: ACTIVE
│ Train: 5,412 | Val: 1,042 | Test: 1,456
│ Patients - Train: 54 | Val: 10 | Test: 17
│ Building model...
└─ Training ───────────────────────────────────────────────┘
Epoch 1/30
170/170 ━━━━━━━━━━━━━━━━━━━━ 426s 2s/step - accuracy: 0.7606 - loss: 0.5902 - val_accuracy: 0.8551 - val_loss: 0.3256 - learning_rate: 0.0010
Epoch 2/30
170/170 ━━━━━━━━━━━━━━━━━━━━ 26s 150ms/step - accuracy: 0.8510 - loss: 0.3806 - val_accuracy: 0.9184 - val_loss: 0.1931 - learning_rate: 0.0010
Epoch 3/30
170/170 ━━━━━━━━━━━━━━━━━━━━ 25s 147ms/step - accuracy: 0.8942 - loss: 0.2665 - val_accuracy: 0.9117 - val_loss: 0.2358 - learning_rate: 0.0010
Epoch 4/30
170/170 ━━━━━━━━━━━━━━━━━━━━ 25s 147ms/step - accuracy: 0.9098 - loss: 0.2321 - val_accuracy: 0.8301 - val_loss: 0.4201 - learning_rate: 0.001

In [4]:
import pandas as pd

# Load and display all results
print("="*70)
print("EXPERIMENT 1: ADAPTATION STUDY")
print("="*70)
exp1 = pd.read_csv('/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/results/experiment1_adaptation.csv')
print(exp1.groupby('cr_level')[['accuracy', 'f1_score', 'auc_roc', 'energy_kwh', 'co2_emissions_g']].mean())

print("\n" + "="*70)
print("EXPERIMENT 2: ROBUSTNESS STUDY")
print("="*70)
exp2 = pd.read_csv('/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/results/experiment2_robustness.csv')
print(exp2.groupby('test_cr')[['accuracy', 'f1_score', 'auc_roc']].mean())

print("\n" + "="*70)
print("EXPERIMENT 3: COMPRESSION-AWARE TRAINING")
print("="*70)
exp3 = pd.read_csv('/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/results/experiment3_compression_aware.csv')
print(exp3.groupby('test_cr')[['accuracy', 'f1_score', 'auc_roc']].mean())

print("\n" + "="*70)
print("SUSTAINABILITY ANALYSIS")
print("="*70)
sust = pd.read_csv('/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/results/sustainability_analysis.csv')
print(sust)

print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)
import json
with open('/content/drive/MyDrive/BreakHis_PLOS_ONE_FINAL/experiment_summary.json', 'r') as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

EXPERIMENT 1: ADAPTATION STUDY
              accuracy  f1_score   auc_roc  energy_kwh  co2_emissions_g
cr_level                                                               
CR100         0.824321  0.877612  0.900668    0.014194         9.813888
CR1000        0.786312  0.844656  0.857722    0.012662         8.754614
CR15          0.830524  0.884842  0.888845    0.012264         8.479660
CR1_original  0.856117  0.897139  0.907441    0.012901         8.919583
CR25          0.844946  0.891108  0.886217    0.013378         9.249823
CR50          0.842745  0.891000  0.896694    0.016338        11.296400
CR500         0.806124  0.863512  0.875922    0.013442         9.293959

EXPERIMENT 2: ROBUSTNESS STUDY
              accuracy  f1_score   auc_roc
test_cr                                   
CR100         0.809693  0.857796  0.906096
CR1000        0.614839  0.666918  0.761680
CR15          0.825109  0.873066  0.907505
CR1_original  0.828061  0.875645  0.909474
CR25          0.823484  0.87144

In [2]:
# Run this cell to diagnose and fix the CR1000 issue

import os
import pandas as pd

# Check CR1000 folder
cr1000_path = '/content/drive/MyDrive/BreakHis_Split/CR1000'

print("=" * 60)
print("DIAGNOSING CR1000 ISSUE")
print("=" * 60)

# Check if folder exists
if not os.path.exists(cr1000_path):
    print(f"❌ CR1000 folder does NOT exist at: {cr1000_path}")
    print("\nYou need to create CR1000 compressed images first.")
else:
    print(f"✅ CR1000 folder exists")

    # Check structure
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(cr1000_path, split)
        if os.path.exists(split_path):
            for class_name in ['benign', 'malignant']:
                class_path = os.path.join(split_path, class_name)
                if os.path.exists(class_path):
                    # Count images
                    total_images = 0
                    for mag in os.listdir(class_path):
                        mag_path = os.path.join(class_path, mag)
                        if os.path.isdir(mag_path):
                            images = [f for f in os.listdir(mag_path) if f.endswith('.png')]
                            total_images += len(images)
                            if len(images) > 0:
                                print(f"  {split}/{class_name}/{mag}: {len(images)} images")
                                # Check first image filename format
                                sample = images[0]
                                print(f"    Sample filename: {sample}")

    # Try loading data
    print("\n" + "=" * 60)
    print("ATTEMPTING TO LOAD CR1000 DATA")
    print("=" * 60)

    images = []
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(cr1000_path, split)
        if not os.path.exists(split_dir):
            print(f"  ❌ Missing: {split}")
            continue

        for class_idx, class_name in enumerate(['benign', 'malignant']):
            class_dir = os.path.join(split_dir, class_name)
            if not os.path.exists(class_dir):
                print(f"  ❌ Missing: {split}/{class_name}")
                continue

            for mag in os.listdir(class_dir):
                mag_dir = os.path.join(class_dir, mag)
                if not os.path.isdir(mag_dir):
                    continue

                for filename in os.listdir(mag_dir):
                    if filename.endswith('.png'):
                        # Try to extract patient ID
                        parts = filename.replace('.png', '').split('-')
                        if len(parts) >= 3:
                            patient_id = f"{parts[1]}-{parts[2]}"
                        else:
                            patient_id = 'unknown'
                            print(f"  ⚠️ Unusual filename format: {filename}")

                        images.append({
                            'filepath': os.path.join(mag_dir, filename),
                            'filename': filename,
                            'class_name': class_name,
                            'class_idx': class_idx,
                            'patient_id': patient_id,
                            'magnification': mag,
                        })

    if len(images) > 0:
        df = pd.DataFrame(images)
        print(f"\n✅ Loaded {len(df)} images from {df['patient_id'].nunique()} patients")
        print(f"\nPatient IDs found: {sorted(df['patient_id'].unique())[:10]}...")
    else:
        print("\n❌ No images found in CR1000!")

DIAGNOSING CR1000 ISSUE
✅ CR1000 folder exists
  train/benign/40X: 467 images
    Sample filename: SOB_B_A-14-22549AB-40-002.png
  train/benign/400X: 431 images
    Sample filename: SOB_B_A-14-22549AB-400-001.png
  train/benign/200X: 449 images
    Sample filename: SOB_B_A-14-22549AB-200-001.png
  train/benign/100X: 471 images
    Sample filename: SOB_B_A-14-22549AB-100-004.png
  train/malignant/100X: 964 images
    Sample filename: SOB_M_MC-14-13413-100-004.png
  train/malignant/400X: 821 images
    Sample filename: SOB_M_MC-14-13413-400-002.png
  train/malignant/200X: 955 images
    Sample filename: SOB_M_MC-14-13413-200-004.png
  train/malignant/40X: 933 images
    Sample filename: SOB_M_MC-14-13413-40-001.png
  val/benign/40X: 50 images
    Sample filename: SOB_B_F-14-29960AB-40-004.png
  val/benign/200X: 51 images
    Sample filename: SOB_B_F-14-29960AB-200-005.png
  val/benign/400X: 47 images
    Sample filename: SOB_B_F-14-29960AB-400-001.png
  val/benign/100X: 49 images
    Sam

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
